# Smart Retinal Camera AI

Research prototype for PhD discussion. Original experiments and outputs preserved. Read AUDIT_AND_REPRODUCIBILITY.md before execution. One completed epoch; no clinical validation.


## Environment and authorised dataset access

Do not embed credentials. Runtime versions are recorded; the original training seed was not fixed.

Original cell indices start at 0.


In [ ]:

import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


In [ ]:

!pip install -q --upgrade kaggle

!kaggle --version

from getpass import getpass
from pathlib import Path

token = getpass("Enter your new Kaggle API token: ")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

token_file = kaggle_dir / "access_token"
token_file.write_text(token.strip())
token_file.chmod(0o600)

del token

print("Kaggle authentication configured.")



In [ ]:

!kaggle competitions download \
    -c aptos2019-blindness-detection \
    -p /content/aptos



In [ ]:

import os
import zipfile

dataset_dir = "/content/aptos"

print("Downloaded files:")
print(os.listdir(dataset_dir))

for filename in os.listdir(dataset_dir):
    if filename.endswith(".zip"):
        zip_path = os.path.join(dataset_dir, filename)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(dataset_dir)

        print("Extracted:", filename)

print("Dataset contents:")
print(os.listdir(dataset_dir))


## Dataset and binary target

Grade 0 versus grades 1–4; not a clinical referral endpoint.

Original cell indices start at 4.


In [ ]:

import pandas as pd

csv_path = "/content/aptos/train.csv"

df = pd.read_csv(csv_path)

print("Total labelled images:", len(df))
print("\nFirst five records:")
display(df.head())

print("\nClass distribution:")
print(df["diagnosis"].value_counts().sort_index())


In [ ]:

df["binary_label"] = (df["diagnosis"] > 0).astype(int)

print("Binary classification distribution:")
print(
    df["binary_label"]
    .value_counts()
    .rename(index={0: "No DR", 1: "DR"})
)


In [ ]:

import os
import cv2
import matplotlib.pyplot as plt

image_dir = "/content/aptos/train_images"

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for grade in range(5):
    sample = df[df["diagnosis"] == grade].iloc[0]
    image_path = os.path.join(
        image_dir, sample["id_code"] + ".png"
    )

    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(image_path)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    axes[grade].imshow(image)
    axes[grade].set_title(f"DR Grade {grade}")
    axes[grade].axis("off")

plt.tight_layout()
plt.show()


## Stratified image-level splits

2,563/549/550; image-ID separation does not establish patient separation.

Original cell indices start at 7.


In [ ]:

from sklearn.model_selection import train_test_split
import os

# First split: 70% training, 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["diagnosis"]
)

# Second split: 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["diagnosis"]
)

# Verify no duplicate image IDs across subsets
train_ids = set(train_df["id_code"])
val_ids = set(val_df["id_code"])
test_ids = set(test_df["id_code"])

assert not (train_ids & val_ids)
assert not (train_ids & test_ids)
assert not (val_ids & test_ids)

assert len(train_df) + len(val_df) + len(test_df) == len(df)

# Save split manifests
os.makedirs("/content/aptos/splits", exist_ok=True)

train_df.to_csv("/content/aptos/splits/train.csv", index=False)
val_df.to_csv("/content/aptos/splits/val.csv", index=False)
test_df.to_csv("/content/aptos/splits/test.csv", index=False)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("Total:", len(df))

print("\nAll image IDs are non-overlapping.")


In [ ]:

for name, subset in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")
    print(
        subset["binary_label"]
        .value_counts(normalize=True)
        .sort_index()
        .round(3)
    )


In [ ]:

import os

print("Dataset exists:",
      os.path.exists("/content/aptos/train_images"))

print("Splits exist:",
      os.path.exists("/content/aptos/splits/train.csv"))


## Exploratory image-quality measurements

Heuristic intensity and edge-detail measures; not validated gradability.

Original cell indices start at 10.


In [ ]:

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load existing training split
train_df = pd.read_csv(
    "/content/aptos/splits/train.csv"
)

# Image quality function
def assess_image_quality(image_path):
    image = cv2.imread(image_path)

    if image is None:
        raise FileNotFoundError(image_path)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    blur = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness = np.mean(gray)
    contrast = np.std(gray)

    return {
        "blur_score": round(float(blur), 2),
        "brightness": round(float(brightness), 2),
        "contrast": round(float(contrast), 2)
    }

# Choose a training image
sample_id = train_df.iloc[0]["id_code"]

image_path = (
    f"/content/aptos/train_images/{sample_id}.png"
)

# Calculate measurements
results = assess_image_quality(image_path)

print("Training images:", len(train_df))
print("Image quality measurements:")
print(results)

# Display retinal image
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(image_rgb)
plt.title(f"Retinal image: {sample_id}")
plt.axis("off")
plt.show()


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load the original retinal image
original = cv2.imread(image_path)

# Simulate loss of camera focus
blurred = cv2.GaussianBlur(original, (21, 21), 5)

# Simulate underexposure
darkened = cv2.convertScaleAbs(
    original, alpha=0.5, beta=0
)

def calculate_metrics(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    return {
        "Blur": round(float(
            cv2.Laplacian(gray, cv2.CV_64F).var()
        ), 2),
        "Brightness": round(float(np.mean(gray)), 2),
        "Contrast": round(float(np.std(gray)), 2)
    }

images = {
    "Original": original,
    "Blurred": blurred,
    "Darkened": darkened
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, img) in zip(axes, images.items()):
    metrics = calculate_metrics(img)

    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"{name}\n"
        f"Blur: {metrics['Blur']}\n"
        f"Brightness: {metrics['Brightness']}\n"
        f"Contrast: {metrics['Contrast']}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

for name, img in images.items():
    print(name, calculate_metrics(img))


In [ ]:

import pandas as pd
import os

# Select 100 reproducible training images
sample_df = train_df.sample(
    n=100,
    random_state=42
)

quality_records = []

for _, row in sample_df.iterrows():
    path = os.path.join(
        "/content/aptos/train_images",
        row["id_code"] + ".png"
    )

    metrics = assess_image_quality(path)

    quality_records.append({
        "id_code": row["id_code"],
        "diagnosis": row["diagnosis"],
        "blur_score": metrics["blur_score"],
        "brightness": metrics["brightness"],
        "contrast": metrics["contrast"]
    })

quality_df = pd.DataFrame(quality_records)

print("Images analysed:", len(quality_df))

display(
    quality_df[
        ["blur_score", "brightness", "contrast"]
    ].describe().round(2)
)


## Model and preprocessing

Frozen ResNet18 backbone; trainable 513-parameter head.

Original cell indices start at 13.


In [ ]:

import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class RetinalDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = os.path.join(
            self.image_dir,
            row["id_code"] + ".png"
        )

        image = Image.open(image_path).convert("RGB")
        label = float(row["binary_label"])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

# Preprocessing for pretrained ResNet18
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

image_dir = "/content/aptos/train_images"

train_dataset = RetinalDataset(
    train_df, image_dir, transform
)

val_dataset = RetinalDataset(
    val_df, image_dir, transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("GPU available:", torch.cuda.is_available())


In [ ]:

import torch
import torch.nn as nn
from torchvision import models

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load pretrained ResNet18
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Freeze pretrained feature extractor
for parameter in model.parameters():
    parameter.requires_grad = False

# Replace final layer for binary classification
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 1)

# Move model to GPU
model = model.to(device)

# Loss function and optimiser
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print("Device:", device)
print("Model:", type(model).__name__)
print("Final layer:", model.fc)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


## Historical setup diagnostics

Preserved incomplete history/5-epoch placeholder. These cells are not evidence of five completed epochs.

Original cell indices start at 15.


In [ ]:

import os

print("Training history exists:",
      "train_losses" in globals())

print("Model exists:",
      "model" in globals())

print("Training checkpoint exists:",
      os.path.exists("/content/aptos/best_resnet18.pth"))



In [ ]:

required = [
    "model",
    "device",
    "train_loader",
    "val_loader",
    "criterion",
    "optimizer"
]

for name in required:
    print(
        name,
        "READY" if name in globals() else "MISSING"
    )


In [ ]:
epochs = 5

train_losses = []
val_losses = []
val_accuracies = []


import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(train_losses, marker="o", label="Training loss")
plt.plot(val_losses, marker="o", label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("ResNet18 training and validation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:

import os

print("Training losses:", train_losses)
print("Validation losses:", val_losses)

print("Number of completed epochs:", len(train_losses))

print(
    "Checkpoint saved:",
    os.path.exists("/content/aptos/best_resnet18.pth")
)


## Completed one-epoch baseline

Only one completed training epoch is supported by the outputs.

Original cell indices start at 19.


In [ ]:

import torch
import time

# Initialise training history
train_losses = []
val_losses = []

start_time = time.time()

# Keep pretrained features frozen
model.eval()
model.fc.train()

running_loss = 0.0
total = 0
correct = 0

print("Starting Epoch 1...", flush=True)

for batch_idx, (images, labels) in enumerate(train_loader):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    logits = model(images).squeeze(1)
    loss = criterion(logits, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

    predictions = (
        torch.sigmoid(logits) >= 0.5
    ).float()

    correct += (predictions == labels).sum().item()
    total += labels.size(0)

    if (batch_idx + 1) % 10 == 0:
        print(
            f"Processed {batch_idx + 1} batches",
            flush=True
        )

train_loss = running_loss / total
train_accuracy = correct / total

train_losses.append(train_loss)

print("\nEpoch 1 completed!")
print("Training loss:", round(train_loss, 4))
print("Training accuracy:", round(train_accuracy, 4))
print("Time:", round(time.time() - start_time, 1), "seconds")


In [ ]:

import torch

model.eval()

running_val_loss = 0.0
correct = 0
total = 0

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

        running_val_loss += loss.item() * images.size(0)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        all_probs.extend(probabilities.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

val_loss = running_val_loss / total
val_accuracy = correct / total

val_losses.append(val_loss)

print("Validation loss:", round(val_loss, 4))
print("Validation accuracy:", round(val_accuracy, 4))
print("Images evaluated:", total)

# Save the current baseline model
torch.save(
    model.state_dict(),
    "/content/aptos/resnet18_epoch1.pth"
)

print("Checkpoint saved successfully.")


## Validation error analysis

Exploratory validation findings; distinct from held-out testing.

Original cell indices start at 21.


In [ ]:

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)
import numpy as np

# Predictions collected during validation
y_true = np.array(all_labels).astype(int)
y_prob = np.array(all_probs)

# Initial decision threshold
threshold = 0.5
y_pred = (y_prob >= threshold).astype(int)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
roc_auc = roc_auc_score(y_true, y_prob)

print("Confusion matrix:")
print(f"True negatives:  {tn}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"True positives:  {tp}")

print("\nPerformance:")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"ROC AUC:     {roc_auc:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["No DR", "DR"]
    )
)


In [ ]:

import pandas as pd
import numpy as np

# Validation predictions were collected without shuffling
analysis_df = val_df.reset_index(drop=True).copy()

analysis_df["prob_dr"] = np.array(all_probs)
analysis_df["prediction"] = (
    analysis_df["prob_dr"] >= 0.5
).astype(int)

# Find photographs with DR that the model missed
false_negatives = analysis_df[
    (analysis_df["binary_label"] == 1) &
    (analysis_df["prediction"] == 0)
].copy()

print("False negatives:", len(false_negatives))

print("\nFalse negatives by DR severity:")
print(
    false_negatives["diagnosis"]
    .value_counts()
    .sort_index()
)

print("\nMissed images:")
display(
    false_negatives[
        ["id_code", "diagnosis", "prob_dr"]
    ].head(10)
)


In [ ]:

import os
import pandas as pd

# Analyse DR-positive validation images only
dr_analysis = analysis_df[
    analysis_df["binary_label"] == 1
].copy()

# Mark each photograph as detected or missed
dr_analysis["result"] = dr_analysis["prediction"].map({
    0: "Missed DR",
    1: "Detected DR"
})

quality_results = []

for _, row in dr_analysis.iterrows():
    path = os.path.join(
        "/content/aptos/train_images",
        row["id_code"] + ".png"
    )

    metrics = assess_image_quality(path)

    quality_results.append({
        "id_code": row["id_code"],
        "diagnosis": row["diagnosis"],
        "result": row["result"],
        "blur_score": metrics["blur_score"],
        "brightness": metrics["brightness"],
        "contrast": metrics["contrast"]
    })

quality_comparison = pd.DataFrame(quality_results)

print("Image quality by classification result:")

display(
    quality_comparison.groupby("result")[
        ["blur_score", "brightness", "contrast"]
    ].agg(["mean", "median", "std"]).round(2)
)


In [ ]:

!nvidia-smi


## Blur at native resolution

Blur before the torchvision transform.

Original cell indices start at 25.


In [ ]:

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score
)

model.eval()

# Evaluate identical images under each condition
experiment_df = val_df.reset_index(drop=True).copy()

conditions = {
    "Original": None,
    "Mild blur": 3,
    "Moderate blur": 7,
    "Strong blur": 15
}

experiment_results = []

for condition, kernel in conditions.items():
    true_labels = []
    probabilities = []

    for _, row in experiment_df.iterrows():
        path = (
            "/content/aptos/train_images/"
            + row["id_code"] + ".png"
        )

        image = cv2.imread(path)

        if image is None:
            raise FileNotFoundError(path)

        # Apply controlled Gaussian blur
        if kernel is not None:
            image = cv2.GaussianBlur(
                image, (kernel, kernel), 0
            )

        image_rgb = cv2.cvtColor(
            image, cv2.COLOR_BGR2RGB
        )

        image_pil = Image.fromarray(image_rgb)
        image_tensor = transform(
            image_pil
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            logit = model(image_tensor).squeeze()
            probability = torch.sigmoid(
                logit
            ).item()

        true_labels.append(
            int(row["binary_label"])
        )
        probabilities.append(probability)

    y_true = np.array(true_labels)
    y_prob = np.array(probabilities)
    y_pred = (y_prob >= 0.5).astype(int)

    experiment_results.append({
        "Condition": condition,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Sensitivity": recall_score(y_true, y_pred),
        "ROC AUC": roc_auc_score(y_true, y_prob)
    })

results_df = pd.DataFrame(experiment_results)

display(results_df.round(4))


In [ ]:

import matplotlib.pyplot as plt

metrics = ["blur_score", "brightness", "contrast"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric in zip(axes, metrics):
    quality_comparison.boxplot(
        column=metric,
        by="result",
        ax=ax
    )
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("")
    ax.set_ylabel("Measured value")

plt.suptitle("")
plt.tight_layout()
plt.show()


## Blur at model resolution

OpenCV resize before blur; baseline differs from PIL/torchvision.

Original cell indices start at 27.


In [ ]:

import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score
)
from torchvision import transforms

model.eval()

# Apply blur after resizing to model resolution
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

blur_conditions = {
    "Original": 0,
    "Mild": 3,
    "Moderate": 9,
    "Strong": 21
}

results = []

for condition, kernel in blur_conditions.items():
    all_predictions = []
    all_targets = []

    batch_images = []
    batch_labels = []

    def process_batch():
        if not batch_images:
            return

        inputs = torch.stack(batch_images).to(device)

        with torch.inference_mode():
            probabilities = torch.sigmoid(
                model(inputs).squeeze(1)
            ).cpu().numpy()

        all_predictions.extend(probabilities.tolist())
        all_targets.extend(batch_labels)

        batch_images.clear()
        batch_labels.clear()

    for _, row in val_df.iterrows():
        path = (
            "/content/aptos/train_images/"
            + row["id_code"] + ".png"
        )

        image = cv2.imread(path)
        if image is None:
            raise FileNotFoundError(path)

        image = cv2.resize(image, (224, 224))

        if kernel > 0:
            image = cv2.GaussianBlur(
                image,
                (kernel, kernel),
                sigmaX=kernel / 3
            )

        image = cv2.cvtColor(
            image, cv2.COLOR_BGR2RGB
        )

        image_tensor = preprocess(
            Image.fromarray(image)
        )

        batch_images.append(image_tensor)
        batch_labels.append(int(row["binary_label"]))

        if len(batch_images) == 32:
            process_batch()

    process_batch()

    y_true = np.array(all_targets)
    y_prob = np.array(all_predictions)
    y_pred = (y_prob >= 0.5).astype(int)

    results.append({
        "Condition": condition,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Sensitivity": recall_score(y_true, y_pred),
        "ROC AUC": roc_auc_score(y_true, y_prob),
        "False negatives": int(
            ((y_true == 1) & (y_pred == 0)).sum()
        )
    })

    print(f"{condition} completed")

robustness_df = pd.DataFrame(results)
display(robustness_df.round(4))


In [ ]:

from sklearn.metrics import confusion_matrix
import pandas as pd

# Reuse the existing robustness experiment code,
# adding confusion-matrix counts inside its loop.

# Within each condition, after calculating y_pred:
tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("Last evaluated condition:", condition)
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)
print("Predicted DR:", int(y_pred.sum()))


In [ ]:

import os
import pandas as pd

output_dir = "/content/aptos/outputs"
os.makedirs(output_dir, exist_ok=True)

# Save results from our second blur experiment
robustness_df.to_csv(
    f"{output_dir}/blur_robustness_results.csv",
    index=False
)

print("Results saved successfully.")
display(robustness_df)


## Grad-CAM illustrations

Positive DR logit explanations; no lesion-localisation validation.

Original cell indices start at 30.


In [ ]:

import torch
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# Restore the saved baseline checkpoint
model.load_state_dict(
    torch.load(
        "/content/aptos/resnet18_epoch1.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

def generate_gradcam(model, input_tensor):
    activations = []
    gradients = []

    def forward_hook(module, inputs, output):
        activations.append(output.detach())

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    # Final convolutional stage of ResNet18
    target_layer = model.layer4[-1]

    forward_handle = target_layer.register_forward_hook(
        forward_hook
    )

    backward_handle = target_layer.register_full_backward_hook(
        backward_hook
    )

    try:
        model.zero_grad(set_to_none=True)

        # Enable gradient tracking for Grad-CAM
        input_tensor = input_tensor.to(device)
        input_tensor = input_tensor.detach().requires_grad_(True)

        logit = model(input_tensor).squeeze()
        logit.backward()

        activation = activations[0]
        gradient = gradients[0]

        weights = gradient.mean(dim=(2, 3), keepdim=True)

        cam = (weights * activation).sum(dim=1)
        cam = F.relu(cam).squeeze(0)

        cam = F.interpolate(
            cam[None, None],
            size=(224, 224),
            mode="bilinear",
            align_corners=False
        ).squeeze()

        cam = cam.cpu().numpy()

        cam = (cam - cam.min()) / (
            cam.max() - cam.min() + 1e-8
        )

        return cam

    finally:
        forward_handle.remove()
        backward_handle.remove()


In [ ]:

# Use a previously identified false-negative image
sample_id = "e25ccfe38e44"

image_path = (
    f"/content/aptos/train_images/{sample_id}.png"
)

original = Image.open(image_path).convert("RGB")

# Use the exact validation preprocessing
input_tensor = transform(original).unsqueeze(0)

cam = generate_gradcam(model, input_tensor)

# Resize original image for display
display_image = np.array(
    original.resize((224, 224))
)

# Create coloured heatmap
heatmap = cv2.applyColorMap(
    np.uint8(255 * cam),
    cv2.COLORMAP_JET
)

heatmap = cv2.cvtColor(
    heatmap, cv2.COLOR_BGR2RGB
)

overlay = cv2.addWeighted(
    display_image,
    0.6,
    heatmap,
    0.4,
    0
)

# Obtain model prediction
with torch.inference_mode():
    logit = model(input_tensor.to(device)).squeeze()
    probability = torch.sigmoid(logit).item()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(display_image)
axes[0].set_title("Original retinal image")

axes[1].imshow(cam, cmap="jet")
axes[1].set_title("Grad-CAM")

axes[2].imshow(overlay)
axes[2].set_title(
    f"Overlay | P(DR)={probability:.3f}"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Image:", sample_id)
print("Model DR score:", round(probability, 4))


In [ ]:

# Find a correctly detected DR image
correct_dr = analysis_df[
    (analysis_df["binary_label"] == 1) &
    (analysis_df["prediction"] == 1)
]

# Select a confident correct prediction
sample = correct_dr.sort_values(
    "prob_dr", ascending=False
).iloc[0]

sample_id = sample["id_code"]

print("Selected image:", sample_id)
print("DR grade:", sample["diagnosis"])
print("Predicted DR score:", round(sample["prob_dr"], 4))

image_path = (
    f"/content/aptos/train_images/{sample_id}.png"
)

original = Image.open(image_path).convert("RGB")
input_tensor = transform(original).unsqueeze(0)

cam = generate_gradcam(model, input_tensor)

display_image = np.array(
    original.resize((224, 224))
)

heatmap = cv2.applyColorMap(
    np.uint8(255 * cam),
    cv2.COLORMAP_JET
)

heatmap = cv2.cvtColor(
    heatmap, cv2.COLOR_BGR2RGB
)

overlay = cv2.addWeighted(
    display_image, 0.6,
    heatmap, 0.4, 0
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(display_image)
axes[0].set_title("Original image")

axes[1].imshow(cam, cmap="jet")
axes[1].set_title("Grad-CAM")

axes[2].imshow(overlay)
axes[2].set_title("Correctly detected DR")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Held-out test evaluation

550 images, threshold 0.5; retained test results.

Original cell indices start at 33.


In [ ]:

import pandas as pd
from torch.utils.data import DataLoader

test_df = pd.read_csv(
    "/content/aptos/splits/test.csv"
)

test_dataset = RetinalDataset(
    test_df,
    "/content/aptos/train_images",
    transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Test images:", len(test_dataset))


In [ ]:

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
    f1_score
)

model.load_state_dict(
    torch.load(
        "/content/aptos/resnet18_epoch1.pth",
        map_location=device,
        weights_only=True
    )
)
model.eval()

test_labels = []
test_probabilities = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device)

        logits = model(images).squeeze(1)
        probabilities = torch.sigmoid(logits)

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )
        test_labels.extend(labels.numpy())

y_true_test = np.array(test_labels).astype(int)
y_prob_test = np.array(test_probabilities)
y_pred_test = (y_prob_test >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true_test,
    y_pred_test,
    labels=[0, 1]
).ravel()

print("HELD-OUT TEST RESULTS")
print("----------------------")
print("Accuracy:", round(accuracy_score(
    y_true_test, y_pred_test
), 4))
print("Sensitivity:", round(tp / (tp + fn), 4))
print("Specificity:", round(tn / (tn + fp), 4))
print("ROC AUC:", round(roc_auc_score(
    y_true_test, y_prob_test
), 4))
print("F1:", round(f1_score(
    y_true_test, y_pred_test
), 4))

print("\nConfusion matrix:")
print("TN:", tn, "FP:", fp)
print("FN:", fn, "TP:", tp)


## Original Drive backup

Historical backup operation retained.

Original cell indices start at 35.


In [ ]:

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

backup_dir = Path(
    "/content/drive/MyDrive/Smart-Retinal-Camera-AI"
)

(backup_dir / "models").mkdir(
    parents=True, exist_ok=True
)
(backup_dir / "splits").mkdir(
    parents=True, exist_ok=True
)
(backup_dir / "results").mkdir(
    parents=True, exist_ok=True
)

# Save trained model
shutil.copy2(
    "/content/aptos/resnet18_epoch1.pth",
    backup_dir / "models/resnet18_epoch1.pth"
)

# Save reproducible dataset splits
for filename in ["train.csv", "val.csv", "test.csv"]:
    shutil.copy2(
        f"/content/aptos/splits/{filename}",
        backup_dir / "splits" / filename
    )

# Save robustness results
shutil.copy2(
    "/content/aptos/outputs/blur_robustness_results.csv",
    backup_dir / "results/blur_robustness_results.csv"
)

print("Backup completed successfully.")
print("Location:", backup_dir)


## Integrated research demonstration

Heuristic review indicators and uncalibrated model scores; not diagnostic.

Original cell indices start at 36.


In [ ]:

import cv2
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

model.eval()

def smart_retinal_camera(image_path):
    # Read the image
    image_bgr = cv2.imread(image_path)

    if image_bgr is None:
        raise FileNotFoundError(image_path)

    # Image quality measurements
    quality = assess_image_quality(image_path)

    # Prepare image for classification
    image_rgb = cv2.cvtColor(
        image_bgr, cv2.COLOR_BGR2RGB
    )

    image_pil = Image.fromarray(image_rgb)

    image_tensor = (
        transform(image_pil)
        .unsqueeze(0)
        .to(device)
    )

    # DR classification
    with torch.inference_mode():
        logit = model(image_tensor).squeeze()
        dr_score = torch.sigmoid(logit).item()

    predicted_class = (
        "DR" if dr_score >= 0.5 else "No DR"
    )

    # Experimental review indicators
    quality_flags = []

    if quality["blur_score"] < 10:
        quality_flags.append("Low edge detail")

    if quality["brightness"] < 40:
        quality_flags.append("Low brightness")

    if quality["brightness"] > 100:
        quality_flags.append("High brightness")

    if quality["contrast"] < 25:
        quality_flags.append("Low contrast")

    # Flag predictions near the decision threshold
    uncertain = 0.4 <= dr_score <= 0.6

    review_required = bool(quality_flags) or uncertain

    return {
        "image": image_rgb,
        "dr_score": round(dr_score, 4),
        "prediction": predicted_class,
        "quality": quality,
        "quality_flags": quality_flags,
        "uncertain_prediction": uncertain,
        "review_required": review_required
    }


In [ ]:

# Select a validation image
sample_id = val_df.iloc[0]["id_code"]

image_path = (
    f"/content/aptos/train_images/{sample_id}.png"
)

result = smart_retinal_camera(image_path)

print("SMART RETINAL CAMERA AI")
print("-----------------------")

print("Image:", sample_id)
print("Prediction:", result["prediction"])
print("DR score:", result["dr_score"])
print("Image quality:", result["quality"])
print("Quality flags:", result["quality_flags"])
print("Uncertain prediction:", result["uncertain_prediction"])
print("Review required:", result["review_required"])

plt.figure(figsize=(6, 6))
plt.imshow(result["image"])
plt.title(
    f"{result['prediction']} | "
    f"DR score: {result['dr_score']}"
)
plt.axis("off")
plt.show()


In [ ]:

import cv2
import os
import pandas as pd

original_path = image_path

original_img = cv2.imread(original_path)

if original_img is None:
    raise FileNotFoundError(original_path)

test_dir = "/content/aptos/demo_images"
os.makedirs(test_dir, exist_ok=True)

# Create degraded images
blurred_img = cv2.GaussianBlur(
    original_img, (31, 31), 8
)

darkened_img = cv2.convertScaleAbs(
    original_img, alpha=0.35, beta=0
)

image_variants = {
    "Original": original_img,
    "Strong blur": blurred_img,
    "Darkened": darkened_img
}

demo_results = []

for name, img in image_variants.items():
    output_path = os.path.join(
        test_dir,
        name.lower().replace(" ", "_") + ".png"
    )

    cv2.imwrite(output_path, img)

    result = smart_retinal_camera(output_path)

    demo_results.append({
        "Condition": name,
        "Prediction": result["prediction"],
        "DR score": result["dr_score"],
        "Blur score": result["quality"]["blur_score"],
        "Brightness": result["quality"]["brightness"],
        "Contrast": result["quality"]["contrast"],
        "Review required": result["review_required"],
        "Flags": ", ".join(result["quality_flags"])
    })

demo_df = pd.DataFrame(demo_results)

display(demo_df)


## Research packaging and audit

See report, execution guide, exported tables and integrity manifest.

Original cell indices start at 39.


# New section

In [ ]:
from google.colab import _message
import json
from pathlib import Path
from datetime import datetime, timezone
project_root = Path('/content/drive/MyDrive/Smart-Retinal-Camera-AI')
assert project_root.is_dir(), 'Existing Drive mount required'
release = project_root / ('research_release_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
release.mkdir()
notebook_snapshot = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
(release / 'original_preserved.ipynb').write_text(json.dumps(notebook_snapshot))
print('Snapshot saved:', release)
print('Cells:', len(notebook_snapshot['cells']))

# Smart Retinal Camera AI: research report

Prepared for a University of Exeter PhD application, 21 November 2026. This is a retrospective, single-dataset research prototype. It is not clinically validated and is not suitable for diagnosis. The name describes an intended camera-support workflow; no physical camera integration, prospective study or deployment performance was demonstrated.

## Research question and contribution
Can a lightweight transfer-learning baseline combine retinal-image classification with image-quality review indicators, and what do controlled degradations reveal about its behaviour? The contribution is the integration and critical evaluation of classification, quality measurements and explanatory visualisations, rather than a claim of a novel network architecture or clinical benefit.

## Data and experimental design
The APTOS 2019 labelled set contains 3,662 photographs. Grades 0–4 contain 1,805, 370, 999, 193 and 295 images respectively. The binary target is grade 0 versus any grade greater than 0: 1,805 No-DR and 1,857 DR images. This is an any-DR label, not a validated referable-DR endpoint.

Two stratified splits using diagnosis grade and random_state=42 produce 2,563 training, 549 validation and 550 test images. Image-ID sets are disjoint. This verifies image-ID separation only; patient-level separation and near-duplicate exclusion have not been established. The labelled internal test split is distinct from Kaggle's unlabelled competition test set.

Images are converted to RGB, resized to 224 by 224 using torchvision/PIL, converted to tensors and normalised using mean [0.485,0.456,0.406] and standard deviation [0.229,0.224,0.225]. A pretrained ResNet18 has frozen backbone parameters and a new 512-to-1 linear head (513 trainable parameters). The backbone remains in evaluation mode during head training, preserving BatchNorm statistics. Training uses BCEWithLogitsLoss, Adam with learning rate 0.001 and batch size 32. Only one completed epoch is evidenced. A separate epochs=5 placeholder does not implement five epochs.

The recorded training loss is 0.4144, training accuracy 0.8697 and elapsed time 277.6 seconds. Validation loss is 0.2969 and accuracy 0.9053. The saved checkpoint is resnet18_epoch1.pth. The binary decision rule is sigmoid score >=0.5; scores have not been calibrated as clinical probabilities.

## Held-out results
The saved test evaluation and exported prediction arrays give 94.00% accuracy, 92.83% sensitivity, 95.20% specificity, ROC AUC 0.9754 and positive-class F1 0.9401. The confusion matrix is TN=258, FP=13, FN=20, TP=259 (271 negatives, 279 positives). Thus 20 positive images were missed. These are point estimates on one internal split, not evidence of clinical safety or superiority to other systems. No confidence intervals, repeated-seed estimates or external validation were originally reported.

Validation results must not be substituted for test results: the initial validation confusion matrix was TN=248, FP=23, FN=29, TP=249, with AUC 0.9613. The 29 missed positive validation images had grade counts 13, 12, 2 and 2 for grades 1–4 respectively. These are exploratory validation findings, not test subgroup performance.

## Controlled blur and quality findings
The first blur experiment applies Gaussian kernels 3, 7 and 15 at native resolution before the standard transform. Original/mild/moderate/strong accuracies were 0.9053/0.9107/0.9071/0.9071 and AUCs 0.9613/0.9612/0.9608/0.9598. Small native-resolution kernels followed by resizing produced little change in this experiment; this does not establish blur robustness.

The second experiment first resizes with OpenCV to 224 by 224, then applies kernels 3, 9 and 21 with sigma=kernel/3. Original/mild/moderate/strong accuracies were 0.9126/0.9071/0.8889/0.8251; sensitivities 0.9065/0.8957/0.8741/0.9173; AUCs 0.9619/0.9579/0.9377/0.9121; false negatives 26/29/35/23. Strong blur had TN=198, FP=73, FN=23, TP=255. Its higher sensitivity therefore accompanies many false positives and lower accuracy/AUC; it is not an improvement in overall robustness. Its specificity, derived from those counts, is 198/271=73.06%.

The unblurred baseline differs between experiments because the resizing pipelines differ (PIL/torchvision versus OpenCV). Compare conditions within each experiment; a matched preprocessing ablation is still needed.

Quality measures are grayscale Laplacian variance, mean intensity and intensity standard deviation. In the 100-image training sample, their means were 25.10, 64.69 and 40.29. Missed positive validation images had higher mean Laplacian variance (26.45) than detected positives (11.64), so this evidence does not support a simple claim that lower edge detail explains the misses. Severity, retinal structure, resolution and borders may confound these measurements; no causal analysis was performed.

## Demonstration and interpretability
The demonstration integrates classification and review flags. Thresholds are edge-detail score <10, brightness <40 or >100, contrast <25, or sigmoid score in [0.4,0.6]. These are heuristic flags, not validated gradability or calibrated uncertainty estimates. A flag does not suppress the model output, and no flag does not imply safety.

For validation image f4c7ae514c54, original, blurred (31 by 31, sigma 8) and darkened (intensity multiplier 0.35) variants all retained the experimental No-DR class, with scores 0.1325, 0.1158 and 0.2121. Review flags were false, true and true respectively. Blur produced the low-edge-detail flag; darkening produced low-edge-detail, low-brightness and low-contrast flags. This is one illustrative image, not a quantitative evaluation of review-rule sensitivity.

Grad-CAM targets the final ResNet18 convolutional block and the positive DR logit, including when the predicted class is negative. The recorded false-negative example e25ccfe38e44 has score 0.2734; the selected confident correct grade-4 example cc3d2e961768 has score 0.9870. These selected illustrations do not validate lesion localisation, causality or explanation reliability. Original embedded panels are preserved privately.

## Reproducibility audit and limitations
Original code and outputs are retained. Empty history plots and checkpoint-existence diagnostics are historical development artefacts, not successful multi-epoch training. Generic variables such as y_true/y_prob and results are overwritten across experiments; export uses explicitly named test arrays and checks their alignment and confusion matrix. Saved output is evidence of a historical run; a clean-runtime execution remains outstanding.

Split seeds are fixed, but initial head weights and training shuffle are not explicitly seeded in the original run. The original random trajectory cannot be recovered retrospectively. A checkpoint and split manifests enable inference replication; a fresh training run may differ. Full restart-and-run-all replication has not been performed during packaging, and historical outputs were not replaced. Requirements record the active runtime, which is not a portable CUDA lockfile. Credentials and dataset archives are excluded from the public scaffold. Private notebooks and figure exports can contain restricted retinal images and must not be uploaded wholesale to GitHub.

Further limitations include single-dataset selection bias, image-level rather than patient-level splitting, possible duplicate leakage, binary reduction of severity, synthetic degradations that do not reproduce all optical artefacts, unvalidated quality thresholds, selected explanation examples, absent calibration and uncertainty intervals, and absent physical-device, latency, fairness or prospective clinical evaluation.

## Future research
First, establish patient/site grouping where available, audit near-duplicates and preregister a locked evaluation protocol. Repeat training with explicit seeds and immutable configuration, retain raw predictions and calculate appropriate uncertainty intervals. Use validation-only selection for thresholds and calibration, then evaluate once on an untouched external dataset. Match resize/interpolation across robustness baselines and investigate real acquisition degradation, exposure and motion. Validate quality measures against expert gradability labels and assess review coverage versus residual error. Evaluate explanations against suitable annotations and sanity checks. Only after appropriate governance should a physical-camera study examine usability, latency, acquisition quality and prospective performance. Exeter-specific supervisor or laboratory fit should be personalised separately rather than invented.

## Evidence provenance
Primary evidence is the supplied Colab notebook (original zero-based cells 0–38), its saved outputs, the existing checkpoint and runtime arrays. Cells 7–8 define splits, 13–21 define the model/training/validation, 25 and 27–29 define blur experiments, 30–32 define Grad-CAM, 33–34 define test evaluation and 36–38 define the demonstration. Exported CSVs retain numerical precision where available; report values follow displayed rounding. No new model training or clinical experiments were performed in preparing this report.

## Method references

- He, K., Zhang, X., Ren, S., and Sun, J. Deep Residual Learning for Image Recognition. arXiv:1512.03385 (2015). https://arxiv.org/abs/1512.03385
- Selvaraju, R. R., et al. Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization. arXiv:1610.02391; ICCV 2017 / revised IJCV version. https://arxiv.org/abs/1610.02391
- APTOS 2019 Blindness Detection, Kaggle competition source: https://www.kaggle.com/competitions/aptos2019-blindness-detection

These identify upstream methods and data; all project performance claims derive from the supplied notebook, not these references.
